# Resilience Score Recalibration V2

**Task:** P8-SCORE-RECALIBRATION | **Author:** @ml-engineer | **Date:** 2026-10-21

## Problem
V1 scoring had 4 critical calibration issues:
1. **Hard ceiling at 76.72** — no service could reach Excellent (90+)
2. **Hazard exposure capped at 70** — step function with only 8 discrete values
3. **Zero SLA differentiation** — premium_plus avg 49.73, standard avg 50.20 (random noise)
4. **Telemetry health max ~52** — raw alarm counts structurally suppress this component

## V2 Formula
`overall_score = 0.4 × path_diversity + 0.3 × hazard_exposure_v2 + 0.3 × telemetry_health_v2 + sla_boost`

| Component | V1 | V2 | Method |
| --- | --- | --- | --- |
| path_diversity | 26.3-100 | 26.3-100 (unchanged) | SRLG-based diversity score |
| hazard_exposure | 0-70 step function | 0-100 continuous | `100 × exp(-active_hazards × ln(2) / 20)` — half-life 20 hazards |
| telemetry_health | 0-52 raw | 0-100 percentile | `100 × (1 - PERCENT_RANK(active_alarms))` |
| SLA boost | none | +5/+2/0 | premium_plus +5, premium +2, standard 0 |

## Results
| Metric | V1 | V2 |
| --- | --- | --- |
| Score range | 10.65-76.72 | 17.94-98.70 |
| Avg score | 49.96 | 66.61 |
| Excellent (90+) | 0 | 314 |
| SLA spread | 0.47 pts | 5.15 pts |

**MLflow Experiment:** `/Users/stephen.hage@databricks.com/provable-resilience-intelligence/ml_score_recalibration`

In [0]:
%sql
-- V2 Resilience Score Recalibration
-- Materializes recalibrated scores into resilience_score_snapshot
-- Run this notebook to re-score all services after data pipeline updates

WITH service_data AS (
    SELECT 
        r.snapshot_id,
        r.service_id,
        r.path_diversity_score,
        r.active_hazards,
        r.active_alarms,
        r.srlg_count,
        r.revenue_at_risk_usd,
        -- FIX 1: Continuous exponential hazard decay (half-life = 20 hazards)
        ROUND(100.0 * EXP(-r.active_hazards * LN(2) / 20.0), 2) as hazard_exposure_v2,
        -- FIX 2: Percentile-based telemetry (fewer alarms = higher score)
        ROUND(100.0 * (1.0 - PERCENT_RANK() OVER (ORDER BY r.active_alarms ASC)), 2) as telemetry_health_v2,
        -- BUG-QA-031 FIX: Use service-level SLA tier (authoritative per-service),
        -- not customer-level SLA tier (66.6% misattributed: 792 vs 2560 premium_plus)
        COALESCE(s.sla_tier, 'standard') as sla_tier
    FROM cdm_tmforum.provable_resilience_intelligence.resilience_score_snapshot r
    JOIN cdm_tmforum.provable_resilience_intelligence.service_inventory s 
        ON r.service_id = s.service_id
    -- Removed LEFT JOIN to customer table — sla_tier now sourced from service_inventory
),
scored AS (
    SELECT *,
        -- Base composite: 0.4/0.3/0.3 weights
        ROUND(0.4 * path_diversity_score + 0.3 * hazard_exposure_v2 + 0.3 * telemetry_health_v2, 2) as base_score,
        -- FIX 3: SLA tier boost
        CASE sla_tier
            WHEN 'premium_plus' THEN 5.0
            WHEN 'premium' THEN 2.0
            ELSE 0.0
        END as sla_boost
    FROM service_data
)
SELECT 
    snapshot_id,
    service_id,
    LEAST(100.0, ROUND(base_score + sla_boost, 2)) as overall_score_v2,
    path_diversity_score,
    hazard_exposure_v2,
    telemetry_health_v2,
    sla_tier,
    sla_boost,
    srlg_count,
    active_hazards,
    active_alarms,
    revenue_at_risk_usd,
    -- Score band
    CASE 
        WHEN LEAST(100.0, base_score + sla_boost) >= 90 THEN 'A: Excellent'
        WHEN LEAST(100.0, base_score + sla_boost) >= 70 THEN 'B: Good'
        WHEN LEAST(100.0, base_score + sla_boost) >= 50 THEN 'C: Moderate'
        WHEN LEAST(100.0, base_score + sla_boost) >= 30 THEN 'D: At Risk'
        ELSE 'F: Critical'
    END as score_band
FROM scored
ORDER BY overall_score_v2 DESC

In [0]:
%sql
-- Validate V2 score distribution by band and SLA tier
SELECT 
    CASE 
        WHEN overall_score >= 90 THEN 'A: Excellent (90+)'
        WHEN overall_score >= 70 THEN 'B: Good (70-89)'
        WHEN overall_score >= 50 THEN 'C: Moderate (50-69)'
        WHEN overall_score >= 30 THEN 'D: At Risk (30-49)'
        ELSE 'F: Critical (<30)'
    END as band,
    COUNT(*) as services,
    ROUND(AVG(overall_score), 2) as avg_score,
    CONCAT('$', FORMAT_NUMBER(SUM(revenue_at_risk_usd), 0)) as revenue_at_risk,
    ROUND(MIN(overall_score), 2) as min_score,
    ROUND(MAX(overall_score), 2) as max_score
FROM cdm_tmforum.provable_resilience_intelligence.resilience_score_snapshot
GROUP BY 1
ORDER BY 1

In [0]:
%sql
-- Verify SLA tier differentiation (premium_plus > premium > standard)
-- BUG-QA-031 FIX: Use service_inventory.sla_tier (authoritative per-service)
SELECT 
    s.sla_tier,
    COUNT(*) as services,
    ROUND(AVG(r.overall_score), 2) as avg_score,
    ROUND(STDDEV(r.overall_score), 2) as stddev,
    ROUND(AVG(r.path_diversity_score), 2) as avg_path_div,
    ROUND(AVG(r.hazard_exposure_score), 2) as avg_hazard,
    ROUND(AVG(r.telemetry_health_score), 2) as avg_telemetry
FROM cdm_tmforum.provable_resilience_intelligence.resilience_score_snapshot r
JOIN cdm_tmforum.provable_resilience_intelligence.service_inventory s ON r.service_id = s.service_id
GROUP BY s.sla_tier
ORDER BY avg_score DESC